# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_json = dataset.metadata.to_json()
print(f"Dataset name: {metadata_json['name']}")
print(f"Description: {metadata_json['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Explore record sets available via Croissant schema.
record_sets = dataset.record_sets
print(f"Available record sets (by @id):")
for rs in record_sets:
    print(f"@id: {rs['@id']} | name: {rs.get('name', '<no name>')}")

# For each record set, list fields (columns) by their @id
for rs in record_sets:
    print(f"\nFields for record set: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for fld in fields:
        if isinstance(fld, dict):
            print(f" - @id: {fld.get('@id', '?')} | name: {fld.get('name', '?')}")
        else:
            print(f" - @id: {fld}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Specify record sets by @id. For this dataset, it's common to have a main table. Find that record set's @id from above.
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print(f"Extracting DataFrames for record sets: {record_set_ids}")
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show columns for the first record set
if len(record_set_ids) > 0:
    first_rs = record_set_ids[0]
    print(f"\nColumns in '{first_rs}':")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Pick a numeric field to analyze (use field @id).
# You can inspect columns in the DataFrame to choose.
target_record_set = record_set_ids[0]
df = dataframes[target_record_set]

# Try to find a likely numeric field (e.g., 'coverage', 'peptide_count', 'MW') - replace with the actual @id
possible_numeric_fields = [col for col in df.columns if df[col].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric fields found for analysis: {possible_numeric_fields}")
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]  # Use the first numeric field
else:
    numeric_field = df.columns[0]  # Fallback

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold} (mean): {len(filtered_df)} records")

# Normalize the selected numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Pick a possible group field (categorical)
categorical_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
group_field = categorical_fields[0] if categorical_fields else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"Grouped mean {numeric_field} by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualization Example: Plot histogram of the numeric field
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If categorical group field exists, plot comparison
if group_field:
    plt.figure(figsize=(12, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded metadata and records using `mlcroissant` via the dataset Croissant schema URL.
- Explored available record sets and identified numeric/categorical fields for analysis.
- Demonstrated basic filtering, normalization, group statistics, and plotted distributions.
- For deeper analysis, refer to the `mlcroissant` documentation and schema details to select specific fields and record sets relevant to biological questions of interest in this dataset.